# Chapter 6: DPO — Direct Preference Optimization

> **Book:** Agentic AI Engineering  
> **Part:** 2 — RL Methods  
> **System:** Guardian Angel System (GAS) — Multi-agent diabetic patient supervision

In [ ]:
# Install dependencies
# !pip install numpy matplotlib

In [ ]:
import os
TEST_MODE = os.getenv("NOTEBOOK_TEST_MODE", "0") == "1"
print(f"TEST_MODE: {TEST_MODE}")

---
## 🎯 1. Chapter Goal

By the end of this notebook you will:
- Understand **DPO's closed-form objective** and why it eliminates the need for a reward model
- Build a **medical advice preference dataset** with 15 (prompt, chosen, rejected) pairs
- Compute the **DPO loss manually** showing the full math
- Understand how **TRL's DPOTrainer** would be configured for the GAS Advisor
- Visualize **reward margins** between chosen and rejected responses

---
## 📖 2. Theory Recap

### The Problem with RLHF

Standard RLHF requires:
1. Train a **reward model** on human preferences
2. Run **PPO** to optimize the policy against the reward model
3. Maintain a **reference model** to prevent reward hacking

This is expensive, unstable, and requires careful hyperparameter tuning.

### DPO's Key Insight

DPO (Rafailov et al., 2023) shows that the optimal policy under the RLHF objective has a **closed form**:

$$\pi^*(y|x) \propto \pi_{\text{ref}}(y|x) \exp\left(\frac{r(x,y)}{\beta}\right)$$

This means we can **reparametrize the reward** in terms of the policy:

$$r(x,y) = \beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x)$$

Substituting into the Bradley-Terry preference model gives the **DPO loss**:

$$\mathcal{L}_{\text{DPO}}(\pi_\theta) = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \left[ \log \sigma\left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right) \right]$$

Where:
- $y_w$ = chosen (winning) response
- $y_l$ = rejected (losing) response  
- $\beta$ = temperature controlling deviation from reference
- $\sigma$ = sigmoid function

### Why DPO Works

| Property | RLHF+PPO | DPO |
|----------|----------|-----|
| Reward model needed | ✅ Yes | ❌ No |
| Training stability | ⚠️ Fragile | ✅ Stable |
| Memory overhead | High (4 models) | Lower (2 models) |
| Hyperparameter sensitivity | High | Low |
| Performance | Strong | Competitive |

---
## 🔨 3. Build It

### 3.1 Medical Advice Preference Dataset

In [ ]:
# Medical advice preference dataset for GAS Advisor alignment
# Format: {prompt, chosen (safe/correct), rejected (unsafe/incorrect)}

preference_dataset = [
    {
        "prompt": "Patient glucose 280 mg/dL, what to do?",
        "chosen": "Check for ketones, contact doctor if >300, drink water, light walk if feeling well",
        "rejected": "Take extra insulin immediately without checking"
    },
    {
        "prompt": "Patient missed their metformin dose this morning. Should they double up?",
        "chosen": "Skip the missed dose and take the next scheduled dose. Never double up metformin as it increases GI side effects.",
        "rejected": "Take double the dose now to compensate for the missed one."
    },
    {
        "prompt": "Glucose reading is 55 mg/dL and patient feels shaky. What action?",
        "chosen": "Treat hypoglycemia immediately: 15g fast-acting carbs (juice, glucose tablets), recheck in 15 min, repeat if still <70.",
        "rejected": "Patient should lie down and rest until they feel better."
    },
    {
        "prompt": "Patient with Type 2 diabetes wants to start intermittent fasting. Advice?",
        "chosen": "Consult endocrinologist first. IF can be beneficial but requires medication adjustment, especially for insulin users, to prevent hypoglycemia.",
        "rejected": "Intermittent fasting is great for everyone, just skip breakfast and you'll be fine."
    },
    {
        "prompt": "HbA1c came back at 9.8%. Patient asks if this is serious.",
        "chosen": "Yes, HbA1c of 9.8% indicates poor glycemic control over the past 3 months. Target is <7%. Urgent medication review and lifestyle intervention needed.",
        "rejected": "It's a bit high but nothing to worry about, just eat less sugar."
    },
    {
        "prompt": "Patient on insulin reports glucose of 400 mg/dL with nausea and vomiting.",
        "chosen": "This may indicate DKA — a medical emergency. Check ketones immediately. If positive, go to ER. Do not delay treatment.",
        "rejected": "Take extra insulin and drink plenty of water, the nausea will pass."
    },
    {
        "prompt": "Patient asks if they can drink alcohol with their diabetes medication.",
        "chosen": "Alcohol can cause hypoglycemia especially with insulin or sulfonylureas. If drinking, eat food, monitor glucose, limit to 1-2 drinks, never drink on empty stomach.",
        "rejected": "Alcohol is fine in moderation, diabetes medication doesn't interact with it."
    },
    {
        "prompt": "Continuous glucose monitor showing rapid drop from 180 to 90 in 30 minutes.",
        "chosen": "Rapid glucose drop warrants attention. Eat 15g carbs preventively, recheck in 15 min. Investigate cause: recent exercise, insulin timing, or meal skipping.",
        "rejected": "90 mg/dL is normal, no action needed."
    },
    {
        "prompt": "Patient with diabetes has a foot wound that isn't healing after 2 weeks.",
        "chosen": "Non-healing diabetic foot wounds are serious. Refer to podiatrist or wound care specialist immediately. Risk of infection and amputation if untreated.",
        "rejected": "Keep it clean and bandaged, it will heal on its own eventually."
    },
    {
        "prompt": "Patient wants to stop insulin because they feel fine and glucose is 140.",
        "chosen": "Never stop insulin without physician guidance. Glucose of 140 may seem acceptable but stopping insulin can cause rapid deterioration. Schedule a medication review.",
        "rejected": "If you feel fine and glucose is 140, you can try stopping insulin for a few days."
    },
    {
        "prompt": "Patient reports frequent nighttime hypoglycemia (glucose <60 at 3am).",
        "chosen": "Nocturnal hypoglycemia is dangerous. Reduce evening basal insulin dose, have a bedtime snack with protein, set CGM alerts. Discuss with endocrinologist.",
        "rejected": "Eat more before bed to prevent it, no need to change medication."
    },
    {
        "prompt": "Patient with diabetes is sick with flu and not eating. Should they skip insulin?",
        "chosen": "Sick day rules: never skip insulin even if not eating. Illness raises glucose. Monitor every 2-4 hours, stay hydrated, check ketones, contact doctor if glucose >300.",
        "rejected": "If you're not eating, skip your insulin dose to avoid hypoglycemia."
    },
    {
        "prompt": "Patient asks about using a friend's insulin pen because they ran out.",
        "chosen": "Never share insulin pens — serious infection risk including HIV and hepatitis. Contact pharmacy for emergency supply or go to urgent care.",
        "rejected": "It should be fine to borrow a pen if it's a new needle, insulin is insulin."
    },
    {
        "prompt": "Glucose is 350 mg/dL before surgery scheduled for tomorrow.",
        "chosen": "Notify surgical team immediately. Surgery with glucose >250 increases infection risk and poor healing. Surgery may need to be postponed until glucose is controlled.",
        "rejected": "Take your regular insulin and proceed with surgery as planned."
    },
    {
        "prompt": "Patient with Type 1 diabetes wants to run a marathon. How to prepare?",
        "chosen": "Consult endocrinologist and sports medicine. Reduce basal insulin 20-30% on race day, carry fast carbs, monitor every 30 min, have glucagon kit available, train with CGM.",
        "rejected": "Just run normally, your body will adapt to the exercise."
    }
]

print(f"Dataset size: {len(preference_dataset)} preference pairs")
print("\nExample pair:")
print(f"  Prompt:   {preference_dataset[0]['prompt']}")
print(f"  Chosen:   {preference_dataset[0]['chosen'][:80]}...")
print(f"  Rejected: {preference_dataset[0]['rejected']}")

### 3.2 DPO Loss — Manual Computation

The DPO loss for a single example:

$$\mathcal{L} = -\log \sigma\left( \beta \cdot \underbrace{\left(\log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)}\right)}_{\text{chosen log-ratio}} - \beta \cdot \underbrace{\left(\log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)}_{\text{rejected log-ratio}} \right)$$

In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def dpo_loss(beta, log_pi_chosen, log_pi_ref_chosen, log_pi_rejected, log_pi_ref_rejected):
    """
    DPO loss: -log(sigmoid(beta * (log_ratio_chosen - log_ratio_rejected)))
    
    Args:
        beta: temperature parameter (typically 0.1-0.5)
        log_pi_chosen: log prob of chosen response under current policy
        log_pi_ref_chosen: log prob of chosen response under reference policy
        log_pi_rejected: log prob of rejected response under current policy
        log_pi_ref_rejected: log prob of rejected response under reference policy
    """
    # Step 1: Compute log-ratios (implicit rewards)
    log_ratio_chosen = log_pi_chosen - log_pi_ref_chosen
    log_ratio_rejected = log_pi_rejected - log_pi_ref_rejected
    
    # Step 2: Compute reward margin
    reward_margin = beta * (log_ratio_chosen - log_ratio_rejected)
    
    # Step 3: DPO loss = -log(sigmoid(reward_margin))
    loss = -np.log(sigmoid(reward_margin))
    
    return loss, reward_margin, log_ratio_chosen, log_ratio_rejected

# Example: First preference pair
# Simulate log probabilities (in practice these come from the model)
beta = 0.1

# Current policy slightly prefers chosen (good alignment)
log_pi_chosen = -2.3      # policy assigns higher prob to chosen
log_pi_ref_chosen = -2.5  # reference was slightly less confident

# Current policy less confident on rejected (good!)
log_pi_rejected = -3.1    # policy assigns lower prob to rejected
log_pi_ref_rejected = -2.8  # reference was more confident on rejected

loss, margin, lr_chosen, lr_rejected = dpo_loss(
    beta, log_pi_chosen, log_pi_ref_chosen, log_pi_rejected, log_pi_ref_rejected
)

print("=" * 50)
print("DPO Loss Computation — Step by Step")
print("=" * 50)
print(f"\nInputs:")
print(f"  beta                = {beta}")
print(f"  log π(y_w|x)        = {log_pi_chosen}")
print(f"  log π_ref(y_w|x)    = {log_pi_ref_chosen}")
print(f"  log π(y_l|x)        = {log_pi_rejected}")
print(f"  log π_ref(y_l|x)    = {log_pi_ref_rejected}")
print(f"\nStep 1 — Log-ratios (implicit rewards):")
print(f"  log-ratio chosen    = {log_pi_chosen} - {log_pi_ref_chosen} = {lr_chosen:.4f}")
print(f"  log-ratio rejected  = {log_pi_rejected} - {log_pi_ref_rejected} = {lr_rejected:.4f}")
print(f"\nStep 2 — Reward margin:")
print(f"  margin = β × (lr_chosen - lr_rejected)")
print(f"         = {beta} × ({lr_chosen:.4f} - {lr_rejected:.4f})")
print(f"         = {margin:.4f}")
print(f"\nStep 3 — DPO Loss:")
print(f"  loss = -log(sigmoid({margin:.4f}))")
print(f"       = -log({sigmoid(margin):.4f})")
print(f"       = {loss:.4f}")
print(f"\n✅ Positive margin ({margin:.4f} > 0) means model prefers chosen — good!")

### 3.3 TRL DPOTrainer Configuration

In [ ]:
# TRL DPOTrainer configuration for GAS Advisor
# This shows how you would configure training — execution skipped in TEST_MODE

trl_config_code = '''
from trl import DPOTrainer, DPOConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset

# Load base model and tokenizer
model_name = "meta-llama/Llama-3.2-3B-Instruct"  # or your GAS base model
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Convert preference dataset to HuggingFace Dataset format
hf_dataset = Dataset.from_list(preference_dataset)

# DPO training configuration
training_args = DPOConfig(
    beta=0.1,                    # KL penalty coefficient
    output_dir="./gas_advisor_dpo",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-7,          # Low LR for fine-tuning
    warmup_ratio=0.1,
    bf16=True,                   # Use bfloat16 for efficiency
    logging_steps=10,
    save_strategy="epoch",
    remove_unused_columns=False,
)

# Initialize DPO trainer
trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=hf_dataset,
    tokenizer=tokenizer,
)

# Train!
trainer.train()
'''

if not TEST_MODE:
    print("DPO Trainer config (not executing — requires GPU and model download):")
    print(trl_config_code)
else:
    print("TEST_MODE: Skipping TRL DPOTrainer execution")
    print("Config shown above would train GAS Advisor with DPO")

print("\nKey hyperparameters:")
print("  beta=0.1      → controls deviation from reference model")
print("  lr=5e-7       → very low LR to preserve base capabilities")
print("  bf16=True     → memory efficient training")

### 3.4 Visualizing Reward Margins

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Simulate log probabilities for all 15 examples
np.random.seed(42)
n = len(preference_dataset)

# Simulate a partially-trained model: mostly prefers chosen but not always
log_pi_chosens = np.random.uniform(-3.5, -1.5, n)
log_pi_ref_chosens = log_pi_chosens + np.random.uniform(-0.3, 0.3, n)
log_pi_rejecteds = np.random.uniform(-4.0, -2.0, n)
log_pi_ref_rejecteds = log_pi_rejecteds + np.random.uniform(-0.3, 0.3, n)

beta = 0.1
margins = []
losses = []

for i in range(n):
    l, m, _, _ = dpo_loss(
        beta,
        log_pi_chosens[i], log_pi_ref_chosens[i],
        log_pi_rejecteds[i], log_pi_ref_rejecteds[i]
    )
    margins.append(m)
    losses.append(l)

# Short labels for x-axis
labels = [f"Ex {i+1}" for i in range(n)]
colors = ['#2ecc71' if m > 0 else '#e74c3c' for m in margins]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Plot 1: Reward margins
bars = ax1.bar(labels, margins, color=colors, edgecolor='black', linewidth=0.5)
ax1.axhline(y=0, color='black', linewidth=1.5, linestyle='--')
ax1.set_title('DPO Reward Margins: Chosen vs Rejected Responses', fontsize=14, fontweight='bold')
ax1.set_ylabel('Reward Margin β(log-ratio_chosen − log-ratio_rejected)', fontsize=10)
ax1.set_xlabel('Preference Example')
green_patch = mpatches.Patch(color='#2ecc71', label='Positive margin (model prefers chosen ✅)')
red_patch = mpatches.Patch(color='#e74c3c', label='Negative margin (model prefers rejected ❌)')
ax1.legend(handles=[green_patch, red_patch])
ax1.grid(axis='y', alpha=0.3)

# Plot 2: DPO losses
ax2.bar(labels, losses, color='#3498db', edgecolor='black', linewidth=0.5)
ax2.set_title('DPO Loss per Example (lower = better alignment)', fontsize=14, fontweight='bold')
ax2.set_ylabel('DPO Loss', fontsize=10)
ax2.set_xlabel('Preference Example')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('dpo_reward_margins.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\nSummary:")
print(f"  Examples with positive margin: {sum(1 for m in margins if m > 0)}/{n}")
print(f"  Mean reward margin: {np.mean(margins):.4f}")
print(f"  Mean DPO loss: {np.mean(losses):.4f}")

---
## 🏥 4. GAS Connection

### DPO in the Guardian Angel System

The **GAS Advisor Agent** is the core reasoning component that generates medical recommendations for diabetic patients. DPO is used to align this agent with safe, evidence-based medical advice.

```
GAS Architecture with DPO:

Patient Data → [Monitor Agent] → [Advisor Agent (DPO-aligned)] → [Safety Critic] → Recommendation
                                        ↑
                              DPO Training Pipeline:
                              - 15+ preference pairs
                              - Safe vs unsafe advice
                              - β=0.1 (conservative alignment)
```

**Why DPO for GAS?**

1. **Safety-critical alignment**: DPO directly optimizes for safe medical advice without reward hacking
2. **No reward model needed**: Reduces system complexity — fewer failure points in a medical system
3. **Stable training**: Medical AI cannot afford training instability that PPO sometimes exhibits
4. **Interpretable**: The reward margin directly shows how much the model prefers safe advice

**The GAS Advisor Agent is aligned with DPO using a preference dataset of safe vs unsafe medical advice.**

The preference dataset covers:
- Hypoglycemia management (glucose <70)
- Hyperglycemia management (glucose >250)
- Medication adherence
- Emergency recognition (DKA, severe hypo)
- Lifestyle guidance (exercise, diet, alcohol)

---
## ✅ 5. Checkpoint

Test your understanding:

**Q1: What does the β parameter control in DPO?**  
A: β controls the KL divergence penalty — how much the trained policy can deviate from the reference model. Higher β = stays closer to reference; lower β = more aggressive alignment.

**Q2: Why doesn't DPO need a separate reward model?**  
A: DPO reparametrizes the reward function in terms of the policy itself, showing that the optimal RLHF policy has a closed form. The reward is implicitly encoded in the log-ratio between policy and reference.

**Q3: What does a positive reward margin mean?**  
A: A positive margin means the model assigns a higher implicit reward to the chosen response than the rejected response — the model is correctly aligned with the preference.

**Q4: How many models does DPO require during training vs RLHF+PPO?**  
A: DPO requires 2 models (policy + reference). RLHF+PPO requires 4 models (policy, reference, reward model, value network).

**Q5: In the GAS context, what makes a response "chosen" vs "rejected"?**  
A: Chosen responses follow evidence-based medical guidelines, recommend consulting healthcare providers, and avoid dangerous self-medication. Rejected responses suggest unsafe actions like doubling insulin doses or ignoring emergency symptoms.